In [1]:
!pip install -q openai

In [2]:
import os
import time
import pandas as pd
from tqdm import tqdm
import re
from sklearn.metrics import classification_report, accuracy_score

from kaggle_secrets import UserSecretsClient
from openai import OpenAI


In [3]:
# Load OpenAI key from Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    OPENAI_API_KEY = user_secrets.get_secret("OPENAI_API_KEY")
except:
    raise RuntimeError("OPENAI_API_KEY not found in Kaggle Secrets")

client = OpenAI(api_key=OPENAI_API_KEY)


In [4]:
import requests

ADMIN_KEY = user_secrets.get_secret("OPENAI_ADMIN_KEY")
PROJECT_ID = "proj_xu6vWFj9DDj7tnFLUqARUlfD"

url = f"https://api.openai.com/v1/organization/projects/{PROJECT_ID}/rate_limits"

headers = {
    "Authorization": f"Bearer {ADMIN_KEY}",
    "Content-Type": "application/json"
}

resp = requests.get(url, headers=headers)
resp.raise_for_status()

rate_limits = resp.json()
rate_limits

{'object': 'list',
 'data': [{'id': 'rl-babbage-002',
   'object': 'project.rate_limit',
   'max_images_per_1_minute': 10,
   'max_requests_per_1_minute': 3000,
   'max_tokens_per_1_minute': 250000,
   'model': 'babbage-002'},
  {'id': 'rl-chatgpt-4o-latest',
   'object': 'project.rate_limit',
   'max_requests_per_1_minute': 200,
   'max_tokens_per_1_minute': 500000,
   'model': 'chatgpt-4o-latest'},
  {'id': 'rl-chatgpt-image-latest',
   'object': 'project.rate_limit',
   'max_images_per_1_minute': 10,
   'max_requests_per_1_minute': 3000,
   'max_tokens_per_1_minute': 250000,
   'model': 'chatgpt-image-latest'},
  {'id': 'rl-dall-e-2',
   'object': 'project.rate_limit',
   'max_images_per_1_minute': 5,
   'max_requests_per_1_minute': 500,
   'max_tokens_per_1_minute': 2147483647,
   'model': 'dall-e-2'},
  {'id': 'rl-dall-e-3',
   'object': 'project.rate_limit',
   'max_images_per_1_minute': 5,
   'max_requests_per_1_minute': 500,
   'max_tokens_per_1_minute': 2147483647,
   'model':

In [5]:
class Config:
    MODEL_NAME = "gpt-4.1"

    POS_DATA_FILE = "/kaggle/input/lrec-tcs-attack-support/sentencePair.txt"
    NEG_DATA_FILE = "/kaggle/input/lrec-tcs-attack-support/sentencePair_neg.txt"

    OUTPUT_FILE = "/kaggle/working/openai_zeroshot_results.csv"
    OUTPUT_FILE2 = "/kaggle/working/openai_fewshot_results.csv"

    SAMPLE_SIZE = 500
    RANDOM_SEED = 42

    TEMPERATURE = 0.0
    MAX_OUTPUT_TOKENS = 2048        # label only
    REQUEST_SLEEP = 0.3           # safe for rate limits

In [6]:
# =============================================================================
#  DATA LOADING & PREPARATION
# =============================================================================
FILENAME_PATTERN = re.compile(r"SupremeCourt_(\d{4})_(\d+)\.txt")

def parse_lrec_line(line: str):
    parts = line.strip().split("\t")
    fname_indices = [i for i, p in enumerate(parts) if p.endswith(".txt")]
    if len(fname_indices) != 2: return None
    try:
        sentpair_id = int(parts[0])
        sent1 = " ".join(parts[fname_indices[0] + 2 : fname_indices[1]]).strip('"')
        sent2 = " ".join(parts[fname_indices[1] + 2 : len(parts) - 2]).strip('"')
        label = parts[-1]
    except (ValueError, IndexError): return None
    return {"id": sentpair_id, "sent1": sent1, "sent2": sent2, "label": label}

def load_lrec_file(filepath: str) -> pd.DataFrame:
    rows = []
    print(f"Reading data from {filepath}...")
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if parsed_row := parse_lrec_line(line):
                rows.append(parsed_row)
    return pd.DataFrame(rows)

def load_and_split_data(config: Config):
    """
    Returns:
      df_test: test dataframe
      df_shots: pool from which few-shot exemplars are drawn (disjoint from df_test)
    """
    print("Loading and splitting data...")
    df_pos = load_lrec_file(config.POS_DATA_FILE)
    df_neg = load_lrec_file(config.NEG_DATA_FILE)
    if df_pos.empty or df_neg.empty:
        print("ERROR: Data files failed to load.")
        return None, None

    # Few-shot pool (sample a chunk to keep memory & speed reasonable)
    df_pos_pool = df_pos.sample(n=min(10000, len(df_pos)), random_state=config.RANDOM_SEED)
    df_neg_pool = df_neg.sample(n=min(10000, len(df_neg)), random_state=config.RANDOM_SEED)
    df_shots = pd.concat([df_pos_pool, df_neg_pool]).reset_index(drop=True)

    # Test set excludes the shot pool to avoid leakage
    remaining_pos = df_pos.drop(df_pos_pool.index)
    remaining_neg = df_neg.drop(df_neg_pool.index)
    df_test = pd.concat([remaining_pos, remaining_neg]).reset_index(drop=True)
    df_test = df_test.dropna(subset=['id', 'sent1', 'sent2', 'label'])
    print(f"Created test set with {len(df_test)} samples.")

    if config.SAMPLE_SIZE > 0:
        print(f"Using a sample of {config.SAMPLE_SIZE} examples from the test set for evaluation.")
        df_test = df_test.sample(n=min(config.SAMPLE_SIZE, len(df_test)), random_state=config.RANDOM_SEED).reset_index(drop=True)

    return df_test, df_shots

In [7]:
def make_zero_shot_prompt(sent1: str, sent2: str) -> str:
    return f"""
You are a legal argument classifier.

Classify the relationship between two sentences as exactly one of:
SUPPORT, ATTACK, or NO_REL.

Sentence 1: "{sent1}"
Sentence 2: "{sent2}"

Output ONLY the label.
"""

In [8]:
def extract_openai_text_and_tokens(response):
    text = response.choices[0].message.content.strip()

    usage = response.usage
    return (
        text,
        {
            "prompt_tokens": usage.prompt_tokens,
            "output_tokens": usage.completion_tokens,
            "total_tokens": usage.total_tokens,
        }
    )

In [9]:
class OpenAIZeroShotClassifier:
    def __init__(self, config: Config):
        self.config = config

    def parse_output(self, text: str) -> str:
        text = text.upper()
        if "SUPPORT" in text:
            return "SUPPORT"
        if "ATTACK" in text:
            return "ATTACK"
        if "NO_REL" in text or "NO RELATION" in text:
            return "NO_REL"
        return "NO_REL"

    def predict(self, df: pd.DataFrame):
        predictions = []
        prompt_tokens = []
        output_tokens = []
        total_tokens = []

        for _, row in tqdm(df.iterrows(), total=len(df)):
            prompt = make_zero_shot_prompt(row.sent1, row.sent2)

            response = client.chat.completions.create(
                model=self.config.MODEL_NAME,
                messages=[
                    {"role": "system", "content": "You classify legal sentence relations."},
                    {"role": "user", "content": prompt},
                ],
                temperature=self.config.TEMPERATURE,
                max_tokens=self.config.MAX_OUTPUT_TOKENS,
            )

            text, usage = extract_openai_text_and_tokens(response)
            pred = self.parse_output(text)

            predictions.append(pred)
            prompt_tokens.append(usage["prompt_tokens"])
            output_tokens.append(usage["output_tokens"])
            total_tokens.append(usage["total_tokens"])

            time.sleep(self.config.REQUEST_SLEEP)

        return predictions, prompt_tokens, output_tokens, total_tokens

In [10]:
def make_few_shot_prompt(sent1, sent2, examples):
    lines = [
        "You are a legal argument classifier.",
        "Classify as SUPPORT, ATTACK, or NO_REL.",
        "Output ONLY the label.\n",
    ]

    for ex in examples:
        lines.append(f'Sentence 1: "{ex["sent1"]}"')
        lines.append(f'Sentence 2: "{ex["sent2"]}"')
        lines.append(f'Label: {ex["label"]}\n')

    lines.append("Now classify:")
    lines.append(f'Sentence 1: "{sent1}"')
    lines.append(f'Sentence 2: "{sent2}"')
    lines.append("Label:")

    return "\n".join(lines)

In [11]:
class OpenAIFewShotClassifier:
    def __init__(self, config: Config, examples: list):
        self.config = config
        self.examples = examples

    def parse_output(self, text: str) -> str:
        text = text.upper()
        if "SUPPORT" in text:
            return "SUPPORT"
        if "ATTACK" in text:
            return "ATTACK"
        if "NO_REL" in text:
            return "NO_REL"
        return "NO_REL"

    def predict(self, df: pd.DataFrame):
        predictions = []
        prompt_tokens = []
        output_tokens = []
        total_tokens = []

        for _, row in tqdm(df.iterrows(), total=len(df)):
            prompt = make_few_shot_prompt(row.sent1, row.sent2, self.examples)

            response = client.chat.completions.create(
                model=self.config.MODEL_NAME,
                messages=[
                    {"role": "system", "content": "You classify legal sentence relations."},
                    {"role": "user", "content": prompt},
                ],
                temperature=self.config.TEMPERATURE,
                max_tokens=self.config.MAX_OUTPUT_TOKENS,
            )

            text, usage = extract_openai_text_and_tokens(response)
            pred = self.parse_output(text)

            predictions.append(pred)
            prompt_tokens.append(usage["prompt_tokens"])
            output_tokens.append(usage["output_tokens"])
            total_tokens.append(usage["total_tokens"])

            time.sleep(self.config.REQUEST_SLEEP)

        return predictions, prompt_tokens, output_tokens, total_tokens

In [12]:
config = Config()
df_test, _ = load_and_split_data(config)

clf = OpenAIZeroShotClassifier(config)
preds, p_tok, o_tok, t_tok = clf.predict(df_test)

true_labels = (
    df_test["label"]
    .str.upper()
    .str.replace(" ", "_")
    .replace({"NO_RELATION": "NO_REL"})
)

print("Accuracy:", accuracy_score(true_labels, preds))
print(classification_report(true_labels, preds, zero_division=0))

Loading and splitting data...
Reading data from /kaggle/input/lrec-tcs-attack-support/sentencePair.txt...
Reading data from /kaggle/input/lrec-tcs-attack-support/sentencePair_neg.txt...
Created test set with 20506 samples.
Using a sample of 500 examples from the test set for evaluation.


100%|██████████| 500/500 [07:08<00:00,  1.17it/s]

Accuracy: 0.568
              precision    recall  f1-score   support

      ATTACK       0.49      0.44      0.46       130
      NO_REL       0.76      0.57      0.65       247
     SUPPORT       0.43      0.70      0.54       123

    accuracy                           0.57       500
   macro avg       0.56      0.57      0.55       500
weighted avg       0.61      0.57      0.57       500



In [13]:
out = pd.DataFrame({
    "pair_id": df_test["id"],
    "sentence_1": df_test["sent1"],
    "sentence_2": df_test["sent2"],
    "original_label": df_test["label"],
    "openai_label": preds,
    "prompt_token": p_tok,
    "output_token": o_tok,
    "total_token": t_tok
})

out.to_csv(config.OUTPUT_FILE, index=False)
print(f"Zero-shot results saved to {config.OUTPUT_FILE}")

Zero-shot results saved to /kaggle/working/openai_zeroshot_results.csv


In [14]:
pd.read_csv(config.OUTPUT_FILE).head()

,pair_id,sentence_1,sentence_2,original_label,openai_label,prompt_token,output_token,total_token
0,15136,"It was , however , held that in case the Junio...","Mr. Mukhoty , learned senior counsel appearing...",NO_REL,SUPPORT,172,2,174
1,4842,"However , the charges Nos. II , V and X were e...","He submitted the report on June 8 , 1987 , wit...",ATTACK,NO_REL,121,2,123
2,10249,For the price of goods is affected by many fac...,Advocate General Mancini in the San Giorgio ca...,SUPPORT,SUPPORT,141,2,143
3,9450,Mr. Andley also attempted to argue that the de...,"In support of this plea , Mr. Andley referred ...",NO_REL,SUPPORT,123,2,125
4,15504,Learned counsel for the petitioners emphatical...,49 .,NO_REL,NO_REL,100,2,102


In [15]:
few_shot_examples = [
    {
        "sent1": "The tribunal misdirected itself in law.",
        "sent2": "The appeal is therefore allowed and the case remanded.",
        "label": "SUPPORT"
    },
    {
        "sent1": "The conviction relied on unlawful assembly.",
        "sent2": "The accused was acquitted under Section 148 IPC.",
        "label": "ATTACK"
    },
    {
        "sent1": "The defendant was guilty of perjury.",
        "sent2": "The Companies Act amendment comes into force next month.",
        "label": "NO_REL"
    }
]

clf = OpenAIFewShotClassifier(config, few_shot_examples)
preds, p_tok, o_tok, t_tok = clf.predict(df_test)

print("Accuracy:", accuracy_score(true_labels, preds))
print(classification_report(true_labels, preds, zero_division=0))

100%|██████████| 500/500 [10:19<00:00,  1.24s/it]

Accuracy: 0.566
              precision    recall  f1-score   support

      ATTACK       0.49      0.52      0.50       130
      NO_REL       0.91      0.48      0.63       247
     SUPPORT       0.42      0.80      0.55       123

    accuracy                           0.57       500
   macro avg       0.61      0.60      0.56       500
weighted avg       0.68      0.57      0.58       500



In [16]:
out_fs = pd.DataFrame({
    "pair_id": df_test["id"],
    "sentence_1": df_test["sent1"],
    "sentence_2": df_test["sent2"],
    "original_label": df_test["label"],
    "openai_label": preds,
    "prompt_token": p_tok,
    "output_token": o_tok,
    "total_token": t_tok
})

out_fs.to_csv(config.OUTPUT_FILE2, index=False)
print(f"Few-shot results saved to {config.OUTPUT_FILE2}")

Few-shot results saved to /kaggle/working/openai_fewshot_results.csv


In [17]:
pd.read_csv(config.OUTPUT_FILE2).head()

,pair_id,sentence_1,sentence_2,original_label,openai_label,prompt_token,output_token,total_token
0,15136,"It was , however , held that in case the Junio...","Mr. Mukhoty , learned senior counsel appearing...",NO_REL,SUPPORT,266,2,268
1,4842,"However , the charges Nos. II , V and X were e...","He submitted the report on June 8 , 1987 , wit...",ATTACK,NO_REL,215,2,217
2,10249,For the price of goods is affected by many fac...,Advocate General Mancini in the San Giorgio ca...,SUPPORT,SUPPORT,235,2,237
3,9450,Mr. Andley also attempted to argue that the de...,"In support of this plea , Mr. Andley referred ...",NO_REL,SUPPORT,217,2,219
4,15504,Learned counsel for the petitioners emphatical...,49 .,NO_REL,NO_REL,194,2,196
